In [1]:
import cv2
import os
import psutil
from deepface import DeepFace

# ================== SETTINGS ==================
RESTRICTED_APPS = [
    "chrome.exe",
    "msedge.exe",
    "brave.exe",
    "firefox.exe",
    "opera.exe"
]

SAFE_APP_FOR_CHILD = "wmplayer"   # Windows Media Player

# ================== CLOSE RESTRICTED APPS ==================
def close_restricted_apps():
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            if proc.info['name'] and proc.info['name'].lower() in RESTRICTED_APPS:
                proc.terminate()
        except:
            pass

# ================== AGE VERIFICATION ==================
def verify_age_and_monitor():
    cap = cv2.VideoCapture(0)

    is_child = False
    is_adult = False

    print("\n📷 Webcam started...")
    print("Press Q to stop monitoring\n")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        try:
            result = DeepFace.analyze(
                frame,
                actions=["age"],
                enforce_detection=False
            )

            age = result[0]["age"]

            if age < 18:
                is_child = True
                is_adult = False
                label = f"CHILD ({age} yrs)"
                color = (0, 0, 255)

                # 🔥 REAL BLOCKING
                close_restricted_apps()

            else:
                is_adult = True
                is_child = False
                label = f"ADULT ({age} yrs)"
                color = (0, 255, 0)

            cv2.putText(
                frame, label, (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2
            )

        except Exception:
            pass

        cv2.imshow("Age Based Child Lock (Press Q)", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    return is_child, is_adult

# ================== MAIN PROGRAM ==================
print("\n====== AGE BASED CHILD LOCK SYSTEM ======")
print("1. Open Browser")
print("2. Open Game (Notepad)")
print("3. Open Media Player")

choice = input("Enter your choice: ")

child, adult = verify_age_and_monitor()

if child:
    print("\n🚫 CHILD DETECTED")
    print("🔒 Restricted apps will be CLOSED automatically")
    print("▶ Opening Media Player only")

    os.system(f"start {SAFE_APP_FOR_CHILD}")

elif adult:
    print("\n✅ ADULT DETECTED → FULL ACCESS")

    if choice == "1":
        os.system("start chrome")
    elif choice == "2":
        os.system("start notepad")
    elif choice == "3":
        os.system("start wmplayer")
    else:
        print("Invalid choice")

else:
    print("\n❌ No face detected → Access denied")




====== AGE BASED CHILD LOCK SYSTEM ======
1. Open Browser
2. Open Game (Notepad)
3. Open Media Player


Enter your choice:  1



📷 Webcam started...
Press Q to stop monitoring


✅ ADULT DETECTED → FULL ACCESS
